In [ ]:
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

In [ ]:
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate


In [69]:
from transformers import pipeline , set_seed
from datasets import load_dataset , load_from_disk
import matplotlib.pyplot as plt
from datasets import load_dataset
import pandas as pd

from transformers import AutoModelForSeq2SeqLM , AutoTokenizer
import nltk
from nltk.tokenize  import sent_tokenize

from tqdm import tqdm
import torch

nltk.download('punkt')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [9]:
device = "cuda" if torch.cuda.is_available() else 'cpu'


In [13]:
model_ckpt = "google/pegasus-cnn_dailymail"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

In [14]:
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [18]:
dataset_dialog = load_dataset("knkarthick/dialogsum")

README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/11.3M [00:00<?, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [19]:
dataset_dialog

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [20]:
dataset_dialog['train']['dialogue'][0]

"#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to get a check-up.\n#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.\n#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?\n#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.\n#Person2#: Ok.\n#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?\n#Person2#: Yes.\n#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.\n#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.\n#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.\n#Person2#: Ok, thanks doctor."

In [22]:
dataset_dialog['train']['summary'][0]

"Mr. Smith's getting a check-up, and Doctor Hawkins advises him to have one every year. Hawkins'll give some information about their classes and medications to help Mr. Smith quit smoking."

In [26]:
split_lengths = [len(dataset_dialog[split])for split in dataset_dialog]
print(f'split lengths {split_lengths}')

print(dataset_dialog['test'][1]['dialogue'])
print(dataset_dialog['test'][1]['summary'])

split lengths [12460, 500, 1500]
#Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention all staff... Effective immediately, all office communications are restricted to email correspondence and official memos. The use of Instant Message programs by employees during working hours is strictly prohibited.
#Person2#: Sir, does this apply to intra-office communications only? Or will it also restrict external communications?
#Person1#: It should apply to all communications, not only in this office between employees, but also any outside communications.
#Person2#: But sir, many employees use Instant Messaging to communicate with their clients.
#Person1#: They will just have to change their communication methods. I don't want any - one using Instant Messaging in this office. It wastes too much time! Now, 

In [83]:
def convert_examples_to_features(example_batch):
    input_encodings = tokenizer(example_batch['dialogue'] , max_length = 1024 , truncation = True)

    target_encodings = tokenizer(
        text_target=example_batch["summary"],
        max_length=128,
        truncation=True
    )
    labels = target_encodings["input_ids"]

    
    labels = [
        -100 if token == tokenizer.pad_token_id else token
        for token in labels
    ]
    return {
        'input_ids' : input_encodings['input_ids'],
        'attention_mask' : input_encodings['attention_mask'],
        'labels' : labels
    }

In [38]:
dataset_dialog_pt = dataset_dialog.map(convert_examples_to_features, batched = True)

Map:   0%|          | 0/12460 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [84]:
dataset_dialog_pt

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1500
    })
})

In [88]:
from transformers import DataCollatorForSeq2Seq

seq2seq_data_collector = DataCollatorForSeq2Seq(tokenizer , model = model_pegasus)



In [93]:
from transformers import TrainingArguments, Trainer 

from transformers import TrainingArguments

trainer_args = TrainingArguments(
    output_dir="pegasus-dialog",
    num_train_epochs = 1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=10,

    eval_strategy="steps",
    eval_steps=200,

    save_strategy="steps",
    save_steps=200
)

In [94]:
trainer = Trainer(
    model=model_pegasus,
    args=trainer_args,
    data_collator=seq2seq_data_collector,
    train_dataset=dataset_dialog_pt["train"],
    eval_dataset=dataset_dialog_pt["validation"]
)

In [95]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss
200,34.941617,2.277463
390,32.441055,2.290031


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=390, training_loss=35.362531847831534, metrics={'train_runtime': 4104.5618, 'train_samples_per_second': 3.036, 'train_steps_per_second': 0.095, 'total_flos': 9071766989635584.0, 'train_loss': 35.362531847831534, 'epoch': 1.0})

In [81]:
dataset_dialog_pt["train"]

Dataset({
    features: ['id', 'dialogue', 'summary', 'topic', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 12460
})

In [98]:
sample = dataset_dialog_pt["test"][0]["dialogue"]

inputs = tokenizer(
    sample,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)

inputs = {k: v.to(device) for k, v in inputs.items()}


summary_ids = model_pegasus.generate(
    **inputs,
    max_length=128,
    num_beams=4
)

print("Generated:")
print(tokenizer.decode(summary_ids[0], skip_special_tokens=True))

print("\nReference:")
print(dataset_dialog_pt["test"][0]["summary"])

Generated:
#Person1# asks Ms. Dawson to type an intra-office memorandum to all employees. #Person1# tells Ms. Dawson that the use of Instant Messaging by employees during working hours is strictly prohibited. #Person1# wants employees to change their communication methods.

Reference:
Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore.


In [99]:
trainer.save_model("pegasus-dialog-final")
tokenizer.save_pretrained("pegasus-dialog-final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('pegasus-dialog-final/tokenizer_config.json',
 'pegasus-dialog-final/tokenizer.json')